# LA wildfire impact analysis

Using Sentinel data instead of Landsat data for geospatial analysis, such as wildfire impact studies, offers several advantages based on the specific characteristics of the sensors and their applications:

### Advantages of Sentinel Data:
1. **Higher Spatial Resolution**:
   - Sentinel-2 provides spatial resolution up to 10 meters for visible and near-infrared bands, compared to Landsat's 30-meter resolution for most bands. This allows for more detailed mapping of burn scars and affected areas[1][7].

2. **Shorter Revisit Time**:
   - Sentinel-2 satellites (A and B) have a combined revisit time of 5 days globally, compared to Landsat's 16-day revisit time (8 days with scene overlap). This frequent revisit capability is crucial for monitoring dynamic events like wildfires[1][8].

3. **Cloud Penetration with Sentinel-1**:
   - Sentinel-1 uses Synthetic Aperture Radar (SAR), which can penetrate clouds and operate day or night. This makes it ideal for areas with frequent cloud cover or inconsistent weather conditions, ensuring data availability during critical periods[3][6].

4. **Better Spectral Coverage**:
   - Sentinel-2 has more spectral bands (13 bands) compared to Landsat 8 (11 bands), including dedicated bands for vegetation monitoring (Red Edge) and water content analysis (SWIR). These are particularly useful for assessing post-fire vegetation health and soil conditions[7][8].

5. **Data Fusion Capabilities**:
   - Sentinel-1 and Sentinel-2 can be combined to enhance analysis robustness. For example, SAR data from Sentinel-1 complements optical data from Sentinel-2 by filling gaps caused by cloud cover, enabling continuous monitoring of fire progression[6].

6. **Free and Open Access**:
   - Both Landsat and Sentinel data are freely available, but Sentinel's archive is newer (since 2014) and provides higher temporal frequency, making it easier to access recent data for rapid response scenarios[7].

### Considerations for Landsat Data:
While Sentinel has advantages in resolution and revisit time, Landsat remains valuable for long-term environmental studies due to its decades-long archive dating back to 1972. It provides consistent historical records that are unmatched by Sentinel's shorter archive[1][4].

### Conclusion:
For wildfire impact studies requiring high-resolution, frequent observations, and cloud-independent imaging, Sentinel data is generally preferred over Landsat. However, combining both datasets can leverage their complementary strengths for comprehensive analysis[6][7].

Citations:
[1] https://tierrainsights.buzz/satellite-earth-observation-data-landsat-vs-sentinel-ca5954532353

[2] https://gis.stackexchange.com/questions/187645/sentinel-1a-and-lansat-8-resolution-difference

[3] https://sentinels.copernicus.eu/web/success-stories/-/sentinels-detect-and-monitor-forest-fires

[4] https://landsat.gsfc.nasa.gov/article/data-in-harmony-nasas-harmonized-landsat-and-sentinel-2-project-2/

[5] https://epic.awi.de/49883/1/remotesensing-11-01730.pdf

[6] https://custom-scripts.sentinel-hub.com/custom-scripts/data-fusion/s2_s1_forest_fire_progression/

[7] https://www.linkedin.com/advice/1/what-advantages-disadvantages-using-landsat-sentinel

[8] https://www.gisagmaps.org/landsat-8-sentinel-2-bands/

[9] https://www.nature.com/articles/s41598-019-56967-x

[10] https://ntrs.nasa.gov/citations/20230017735

[11] https://www.mdpi.com/2673-4591/10/1/23

[12] https://documentation.dataspace.copernicus.eu/APIs/openEO/openeo-community-examples/python/ForestFire/ForestFire.html

[13] https://www.usgs.gov/centers/eros/science/usgs-eros-archive-sentinel-2-comparison-sentinel-2-and-landsat

[14] https://www.tandfonline.com/doi/full/10.1080/01431161.2024.2394238

[15] https://www.sciencedirect.com/science/article/pii/S0303243421000544

[16] https://www.tandfonline.com/doi/full/10.1080/15481603.2017.1370169

[17] https://disasters-geoplatform.hub.arcgis.com/pages/california-wildfires-remote-sensing-with-sentinel-2-imagery

[18] https://www.sciencedirect.com/science/article/pii/S2666017221000055

[19] https://www.mdpi.com/2072-4292/16/3/556

[20] https://www.sciencedirect.com/science/article/pii/S0034425719300252


In [9]:
# %pip install censusdata
import pandas as pd
import censusdata
from datetime import datetime
import ee
import geemap
import matplotlib.pyplot as plt


In [45]:
# PARAMETERS — edit these for fire or other environmental event
# California wildfire event
# https://www.fire.ca.gov/incidents/2025/1/7/mountain-fire/

# Define the study area and time periods for the analysis
state_fp = '06'  # California
la_fp = '037'    # Los Angeles county
sd_fp = '073'    # San Diego county
pre_start, pre_end = '2024-12-06', '2025-01-06'
event_start, event_end = '2025-01-07', '2025-01-31'
post_start, post_end = '2025-02-01', '2025-02-28'


In [47]:
# Load U.S. county boundaries from TIGER/2018/Counties to define the ROI
# and filter to the specified state and county
# Note: You can use the following URL to find the state FIPS code:
# https://www.census.gov/library/reference/code-lists/ansi.html
# and the county name:
# https://www.census.gov/geographies/reference-files/time-series/geo/county-administrative.html
# The state FIPS code is a two-digit number, and the county name should be
# the official name as listed in the Census Bureau's reference files.
# Define region of interest
counties = ee.FeatureCollection("TIGER/2018/Counties") \
    .filter(ee.Filter.eq('STATEFP', state_fp)) \
    .filter(ee.Filter.eq('COUNTYFP', la_fp))
union_geom = counties.geometry()


In [48]:
# Improved cloud masking function combining QA60 and SCL approaches
def mask_clouds(image):
    # QA60 cloud mask (all Sentinel-2 products)
    qa_mask = image.select('QA60').bitwiseAnd(0b11 << 10).eq(0)
    
    # SCL-based mask (L2A products)
    scl = image.select('SCL')
    scl_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9))  # Keep 4,5,6,11
    
    return image.updateMask(qa_mask).updateMask(scl_mask)


In [49]:
# Function to build a clean collection with diagnostic output
# Modified collection builder with server-side diagnostics
def get_clean_collection(start_date, end_date, label):
    raw_col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(union_geom) \
        .filterDate(start_date, end_date)
    
    # Server-side count tracking
    count = raw_col.size()
    count_info = count.getInfo()  # Single safe client-side call
    print(f"{label} raw image count: {count_info}")
    
    cloud_filtered = raw_col.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    return cloud_filtered.map(mask_clouds).median()

In [50]:
# Get median composites for each period
print("Building pre-fire composite...")
pre_comp = get_clean_collection(pre_start, pre_end, "Pre-fire")

print("Building event-period composite...")
event_comp = get_clean_collection(event_start, event_end, "Event")

print("Building post-fire composite...")
post_comp = get_clean_collection(post_start, post_end, "Post-fire")



Building pre-fire composite...
Pre-fire raw image count: 49
Building event-period composite...
Event raw image count: 35
Building post-fire composite...
Post-fire raw image count: 41


In [51]:
# Enhanced visualization parameters
# True color with histogram stretch for better contrast
true_color_vis = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 300,  # Adjusted minimum for better contrast
    'max': 2500, # Adjusted maximum based on typical reflectance values
    'gamma': 1.4 # Slightly higher gamma for better visualization
}

# False color composite optimized for burn scar detection (SWIR, NIR, Red)
burn_scar_vis = {
    'bands': ['B12', 'B8', 'B4'], # SWIR, NIR, Red - best for burn scars
    'min': 300,
    'max': 4000, # Higher max for SWIR to show burn scars better
    'gamma': 1.3
}

# Urban burning detection
urban_burn_vis = {
    'bands': ['B12', 'B11', 'B8'],  # SWIR2-SWIR1-NIR
    'min': 200,
    'max': 3000,
    'gamma': 1.5,
}



In [52]:
# Create a map
Map = geemap.Map(basemap="HYBRID")
Map.centerObject(union_geom, 9)
# Add layers to map with visibility control
# if pre_comp is not None:
#     Map.addLayer(pre_comp, true_color_vis, 'Pre-fire True Color')
#     Map.addLayer(pre_comp, burn_scar_vis, 'Pre-fire Burn Detection View', False)
#     Map.addLayer(pre_comp, urban_burn_vis, 'Pre-fire Urban Burn Detection View', False)

if event_comp is not None:
    Map.addLayer(event_comp, true_color_vis, 'During-fire True Color')
    Map.addLayer(event_comp, burn_scar_vis, 'During-fire Burn Detection View')
    Map.addLayer(event_comp, urban_burn_vis, 'During-fire Urban Burn Detection View')

# if post_comp is not None:
#     Map.addLayer(post_comp, true_color_vis, 'Post-fire True Color')
#     Map.addLayer(post_comp, burn_scar_vis, 'Post-fire Burn Detection View')
#     Map.addLayer(post_comp, urban_burn_vis, 'Post-fire Urban Burn Detection View')
Map

Map(center=[34.19572520559816, -118.2617765331474], controls=(WidgetControl(options=['position', 'transparent_…

## Urban-focused Composite Generation

In [33]:
# 2. Date windows around the fire peak (2025‑01‑15)
event = ee.Date('2025-01-15')
pre_start, pre_end   = event.advance(-14, 'day'), event.advance(-1, 'day')
post_start, post_end = event.advance(1, 'day'),  event.advance(14, 'day')

# 3. Build cloud_score by exact system:index join + SCL masking
cloud_prob = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
def add_cloud_score(img):
    prob = cloud_prob \
      .filter(ee.Filter.eq('system:index', img.get('system:index'))) \
      .first()
    score = ee.Image(100).subtract(prob.select('probability'))
    scl   = img.select('SCL')
    bad   = scl.eq(3) .Or(scl.eq(8)) \
               .Or(scl.eq(9)) \
               .Or(scl.eq(10)) \
               .Or(scl.eq(11))  # shadow, cloud, cirrus, snow
    return img.addBands(score.subtract(bad.multiply(100)).rename('cloud_score'))

def clean_mosaic(start, end):
    return (ee.ImageCollection('COPERNICUS/S2_SR')
      .filterBounds(union_geom)
      .filterDate(start, end)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
      .map(add_cloud_score)
      .qualityMosaic('cloud_score')
    )

pre_comp  = clean_mosaic(pre_start,  pre_end)
post_comp = clean_mosaic(post_start, post_end)

# 4. Mask to built‑up via UBI > 0.10
def mask_built(img):
    swir = img.select('B12').multiply(0.0001)
    nir  = img.select('B8').multiply(0.0001)
    ubi  = swir.subtract(nir).divide(swir.add(nir))
    return img.updateMask(ubi.gt(0.10))

pre_built  = mask_built(pre_comp)
post_built = mask_built(post_comp)

# 5. Compute dNBR on the built‑up pixels
def add_nbr(img):
    return img.normalizedDifference(['B8','B12']).rename('NBR')

pre_nbr  = add_nbr(pre_built)
post_nbr = add_nbr(post_built)
dNBR     = pre_nbr.subtract(post_nbr).rename('dNBR')

# 6. Classify burn severity (0–3)
burn_sev = dNBR.expression(
  'dNBR > 0.27 ? 3 : dNBR > 0.10 ? 2 : dNBR > 0 ? 1 : 0',
  {'dNBR': dNBR}
).rename('severity')

# 7. Visualization parameters
rgb_vis = {'bands':['B4','B3','B2'], 'min':0, 'max':3000}
swir_vis= {'bands':['B12','B8','B4'],'min':500,'max':4000}

# Compute 2–98th percentiles for dNBR
stats = dNBR.reduceRegion(
  ee.Reducer.percentile([2,98]),
  union_geom, scale=10, bestEffort=True
)
dnbr_vis = {
  'bands':['dNBR'],
  'min': stats.get('dNBR_p2'),
  'max': stats.get('dNBR_p98'),
  'palette':['green','white','brown']
}
sev_vis = {
  'bands':['severity'],
  'min':0, 'max':3,
  'palette':['#00FF00','#FFFF00','#FFA500','#FF0000']
}

# 8. Display everything
Map = geemap.Map(basemap="HYBRID")
Map.centerObject(union_geom, 10)

# Only the built‑up composites
Map.addLayer(pre_built,  rgb_vis,    'Pre‑fire Built‑up RGB')
Map.addLayer(post_built, swir_vis,   'Post‑fire Built‑up SWIR')

# Change metric
Map.addLayer(dNBR,       dnbr_vis,   'dNBR on Built‑up')

# Burn severity (toggle this off to see the RGB/SWIR beneath)
Map.addLayer(burn_sev,    sev_vis,    'Burn Severity')

Map.addLayerControl()
Map


Map(center=[33.62825041310195, -117.52882798370143], controls=(WidgetControl(options=['position', 'transparent…

In [46]:
# Load U.S. census tracts from TIGER/2020/TRACT for joining with census demographics
tracts = ee.FeatureCollection("TIGER/2020/TRACT") \
    .filterBounds(union_geom)

In [15]:
# 5. VIIRS Black Marble monthly composites 
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
           .select('avg_rad')

viirs_pre  = viirs.filterDate(pre_start, pre_end).mean()
viirs_post = viirs.filterDate(post_start, post_end).mean()
rad_diff   = viirs_pre.subtract(viirs_post).rename('rad_diff')